# Automatic Speech Recognition with Wav2Vec2

### 1. Explore the speech data
LibriSpeech is a corpus of approximately 1000 hours of 16kHz read English speech. The dataset is divided into "clean" (high-quality) and "other" (noisier/less clear) recordings to test model robustness.

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset("librispeech_asr", 
                       "clean", 
                       split="test[:10%]"
)

# dataset = dataset.cast_column("audio", Audio(decode=False))
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

Sanity Check

In [ ]:
print(len(dataset))

Inspect one sample from the dataset

In [ ]:
sample = dataset[0]
sample

In [ ]:
waveform = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]

Let's play the sample

In [ ]:
import IPython.display as ipd

ipd.Audio(waveform, rate=sampling_rate)

### 2. Wav2Vec2

In [ ]:
from transformers import pipeline

asr_wav2vec = pipeline("automatic-speech-recognition",
                    model="facebook/wav2vec2-base-960h",
                    framework="pt")

Run ASR

In [ ]:
prediction = asr_wav2vec(sample["audio"]["array"])
print(prediction["text"])

### 3. Error Analysis

In [ ]:
reference = sample["text"]
predicted = prediction["text"]

print("REF:", reference)
print("HYP:", predicted)

WER Calculation

In [ ]:
from jiwer import wer

error = wer(reference.lower(), predicted.lower())
print("WER:", error)

### 5. Let's add some noises.

In [ ]:
import numpy as np

noisy_waveform = waveform + 0.02 * np.random.randn(len(waveform))

ipd.Audio(noisy_waveform, rate=sampling_rate)

In [ ]:
noisy_prediction = asr_wav2vec(noisy_waveform)
noisy_predicted_text = noisy_prediction["text"]
noisy_predicted_text

In [ ]:
error = wer(reference.lower(), noisy_predicted_text.lower())
print("WER:", error)